# ai_parse_document / ai_prep_search 出力確認ノートブック

このノートブックは Lakeflow パイプライン本体（`transformations/`）とは独立して、
`ai_parse_document` と `ai_prep_search` の**実際の出力スキーマ**を確認するための探索用ノートブックです。

`silver_parsed_documents.py` の `PARSED_TEXT_EXPR` と
`stg_chunks_ai_prep_search.py` の `AI_PREP_SEARCH_EXPR` は、これらの関数の戻り値 struct の
フィールド名を仮置きで実装しています（プレビュー機能のため変更されうるため）。
このノートブックの実行結果を見ながら、必要であれば実装側の式を調整してください。

In [ ]:
dbutils.widgets.text("catalog", "rag_data_platform")
dbutils.widgets.text("schema", "rag_dev")
dbutils.widgets.text("volume_path", "/Volumes/rag_data_platform/rag_dev/raw_documents")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_path = dbutils.widgets.get("volume_path")
print(catalog, schema, volume_path)

## 1. サンプルドキュメントの確認

`seed_sample_data.py` によって UC Volume に配置済みのサンプルドキュメント一覧を確認する。

In [ ]:
display(dbutils.fs.ls(volume_path))

## 2. ai_parse_document の出力確認

サンプルデータは `.txt` のため厳密には対象外だが、`ai_parse_document` が
テキストファイルに対してどう振る舞うか、また PDF/画像を置いた場合の
戻り値 struct の形状を確認するためのセル。PDFサンプルを別途 volume に置いた場合は
`sample_pdf_path` を書き換えて実行する。

In [ ]:
raw_df = (
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load(volume_path)
    .limit(1)
)

parsed_df = raw_df.selectExpr("path", "ai_parse_document(content) AS parsed")
display(parsed_df)

In [ ]:
# 戻り値のスキーマそのものを確認する（struct のフィールド名がここでわかる）
parsed_df.printSchema()

## 3. ai_prep_search の出力確認

`silver_parsed_documents` テーブルが既にパイプラインで作成済みであれば、
そこから本文テキストを取得して `ai_prep_search` の挙動を確認する。
テーブルがまだ無い場合は、サンプルテキストを直接与えて確認する。

In [ ]:
sample_text = """Acme Analytics へようこそ。本ドキュメントは入社1ヶ月目の社員向けに、
社内システムの利用方法と基本的な業務フローをまとめたものである。"""

prep_df = spark.sql(
    "SELECT ai_prep_search(:sample_text) AS chunks",
    args={"sample_text": sample_text},
)
display(prep_df)

In [ ]:
prep_df.printSchema()

## 4. silver_parsed_documents / gold_document_chunks_for_search の中身確認

パイプライン実行後、実際に生成されたテーブルの中身を軽く確認する。

In [ ]:
display(spark.table(f"{catalog}.{schema}.silver_parsed_documents").limit(10))

In [ ]:
display(
    spark.table(f"{catalog}.{schema}.gold_document_chunks_for_search")
    .groupBy("chunk_method", "department", "classification")
    .count()
    .orderBy("chunk_method", "department")
)